# LLM UI Generator — 10K Dataset Pipeline
Run in Google Colab or Kaggle. Each cell saves progress — resume anytime.

In [ ]:
# ── CELL 1: Setup ──────────────────────────────────────────
!pip install requests -q
import os

# SET YOUR API KEYS HERE
os.environ['GEMINI_API_KEY_1'] = 'YOUR_GEMINI_KEY_1_HERE'
os.environ['GEMINI_API_KEY_2'] = 'YOUR_GEMINI_KEY_2_HERE'  # optional 2nd account
os.environ['GROQ_API_KEY_1']   = 'YOUR_GROQ_KEY_1_HERE'
os.environ['GROQ_API_KEY_2']   = 'YOUR_GROQ_KEY_2_HERE'   # optional 2nd account

# Verify keys are set
print('Gemini keys:', len([k for k in [os.environ.get('GEMINI_API_KEY_1',''), os.environ.get('GEMINI_API_KEY_2','')] if k and 'YOUR' not in k]))
print('Groq keys:',   len([k for k in [os.environ.get('GROQ_API_KEY_1',''),   os.environ.get('GROQ_API_KEY_2','')] if k and 'YOUR' not in k]))

# If running in Kaggle, use secrets instead:
# from kaggle_secrets import UserSecretsClient
# secrets = UserSecretsClient()
# os.environ['GEMINI_API_KEY_1'] = secrets.get_secret('GEMINI_API_KEY')
# os.environ['GROQ_API_KEY_1']   = secrets.get_secret('GROQ_API_KEY')

print('Setup done!')

In [ ]:
# ── CELL 2: Stage 1 — Query Generation ────────────────────
# Generates 435 queries per category = 10,005 total
# Takes ~30 min using Groq (fast)
!python stage1_queries.py

In [ ]:
# ── CELL 3: Stage 2 — LLM Responses ───────────────────────
# Generates rich responses for each query
# Takes ~6 hours (API rate limits)
# SAFE TO INTERRUPT — resumes from where it stopped
!python stage2_responses.py

In [ ]:
# ── CELL 4: Stage 3 — A2UI JSON Generation ─────────────────
# Generates rich 40-80 component A2UI JSONs using Gemini
# Takes ~8 hours (Gemini rate limits)
# SAFE TO INTERRUPT — resumes from where it stopped
!python stage3_a2ui_json.py

In [ ]:
# ── CELL 5: Stage 4 — HTML Generation ─────────────────────
# Pure Python — no API needed — done in ~5 minutes
!python stage4_html.py

In [ ]:
# ── CELL 6: Check Status ───────────────────────────────────
import os, json

cats = ['information_retrieval','entertainment','product_lookup','booking','weather',
        'data_visualization','planning','productivity','recipe','localization',
        'technical_support','creative_writing','event_schedule','research_analysis',
        'comparison','calculation','travel','navigation','education','documentation',
        'media_playback','qr_scanner','status_check']

print(f'{"Category":<25} {"Queries":<10} {"Responses":<12} {"Valid JSON":<12} {"HTML"}')
print('-'*70)
tq=tr=tj=th=0
for cat in cats:
    q=r=j=h=0
    qf=f'data/queries/{cat}.json';rf=f'data/responses/{cat}.json';af=f'data/a2ui_json/{cat}.json';hd=f'data/html/{cat}'
    if os.path.exists(qf):
        with open(qf) as f: q=json.load(f).get('count',0)
    if os.path.exists(rf):
        with open(rf) as f: r=json.load(f).get('response_count',0)
    if os.path.exists(af):
        with open(af) as f: j=json.load(f).get('valid_count',0)
    if os.path.exists(hd): h=len([x for x in os.listdir(hd) if x.endswith('.html')])
    print(f'{cat:<25} {q:<10} {r:<12} {j:<12} {h}')
    tq+=q;tr+=r;tj+=j;th+=h
print('-'*70)
print(f'{"TOTAL":<25} {tq:<10} {tr:<12} {tj:<12} {th}')


In [ ]:
# ── CELL 7: Download data (Colab only) ────────────────────
import shutil
shutil.make_archive('dataset_10k', 'zip', 'data')
from google.colab import files
files.download('dataset_10k.zip')